# 09 — Umegaki Relative Entropy

Lift the relative entropy of `02/02` to density matrices. You will compute the **Umegaki relative entropy** $S(\rho\|\sigma) = \mathrm{tr}\,\rho\,(\log_2\rho - \log_2\sigma)$ by hand, watch it give one bit for $|+\rangle$ against $I/2$, whose $Z$ statistics are identical, and check the three properties `02/02` and `02/02b` asked of a quantum lift: never negative (Klein), infinite exactly when the support of $\rho$ is not contained in that of $\sigma$, and never increased by a channel (Lindblad).

**Prerequisites:** `01/07` (a Hermitian matrix has real eigenvalues and an orthonormal eigenbasis), `01/12` (density matrices, projectors), `01/13` (the von Neumann entropy and the $|+\rangle$ versus $I/2$ seed), `01/19` (quantum channels, their Kraus operators, dephasing and depolarizing), `02/02_kl_divergence` (relative entropy, Gibbs' inequality, the infinite value off the support), `02/02b` (the data-processing inequality).

**What you'll be able to do**
- Compute $S(|{+}\rangle\langle{+}|\,\|\,I/2) = 1$ bit by hand from eigenvalues, and recognise $S(\rho\|I/d) = \log_2 d - S(\rho)$.
- Show that two states diagonal in the same basis give the classical $D(p\|q)$.
- State and prove Klein's inequality, and state the exact condition for an infinite value.
- State Lindblad's monotonicity theorem, check it on the channels of `01/19`, and derive its consequences.

You already own every piece; today they click together into the quantity that the entropic measures of this module grow from.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, shannon_entropy
from qot_course.infotheory.quantum import quantum_relative_entropy
from qot_course.quantum.composite import apply_channel, depolarizing_channel, is_cptp
from qot_course.quantum.density import density_from_bloch, density_matrix, maximally_mixed, von_neumann_entropy
from qot_course.quantum.gates import HADAMARD, PAULI_X, PAULI_Z, ry, rz
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS

rng = np.random.default_rng(0)   # random qubit states and rotations for the checks of sections 2 and 3
viz.use_course_style()

## 1. From KL to Umegaki: the one bit, by hand

A density matrix is a probability vector with a basis attached: $\rho = \sum_i p_i\,|i\rangle\langle i|$, eigenvalues $p_i$ (`01/13`) living along eigenvectors $|i\rangle$. If $\sigma = \sum_i q_i\,|i\rangle\langle i|$ has the *same* eigenvectors, a quantum relative entropy should reduce to the classical $D(p\|q)$ of `02/02`. The question is what to do when the eigenvectors differ.

Umegaki's answer (1962) is to keep the formula and let the logarithm act on matrices. **A function of a Hermitian matrix.** Write $A = \sum_i a_i\,|i\rangle\langle i|$ in its orthonormal eigenbasis (`01/07`) and set $f(A) = \sum_i f(a_i)\,|i\rangle\langle i|$: $f$ acts on the eigenvalues and keeps the eigenvectors. You have met two cases, the square root inside the fidelity of `01/14` and the entropy $-\mathrm{tr}\,\rho\log_2\rho = -\sum_i p_i\log_2 p_i$ of `01/13`. By hand, take the state with Bloch vector $(0.6, 0, 0)$: eigenvalue $0.8$ on $|{+}\rangle$ and $0.2$ on $|{-}\rangle$. Since $|{\pm}\rangle\langle{\pm}| = \tfrac12\begin{pmatrix}1 & \pm 1\\ \pm 1 & 1\end{pmatrix}$, $\log_2\rho$ has diagonal entries $\tfrac12(\log_2 0.8 + \log_2 0.2) = -1.3219$ and off-diagonal entries $\tfrac12(\log_2 0.8 - \log_2 0.2) = 1$: not the logarithm of each entry. With the convention $0\log 0 = 0$ of `02/01`, the **Umegaki relative entropy** is

$$
S(\rho\,\|\,\sigma) \;=\; \mathrm{tr}\,\rho\,(\log_2\rho - \log_2\sigma) \;=\; \underbrace{\mathrm{tr}\,\rho\log_2\rho}_{-S(\rho)} \;\underbrace{-\;\mathrm{tr}\,\rho\log_2\sigma}_{\text{cross term}} .
$$

The first term is minus the von Neumann entropy; the second is a cross-entropy. This is `02/02`'s $D(p\|q) = H(p, q) - H(p)$, written with matrices. When $\rho$ and $\sigma$ commute they share an eigenbasis and $S(\rho\|\sigma) = D(p\|q)$ exactly. When they do not, the cross term pairs the eigenvalues of $\rho$ with the logarithms of $\sigma$ in a *different* basis, and that is where the coherence enters.

**The one bit, by hand.** Take $\rho = |{+}\rangle\langle{+}|$ and $\sigma = I/2$.

- $\rho$ is pure: its eigenvalues are $1$ (eigenvector $|{+}\rangle$) and $0$ (eigenvector $|{-}\rangle$). So $\mathrm{tr}\,\rho\log_2\rho = 1\cdot\log_2 1 + 0\cdot\log_2 0 = 0$.
- $\sigma = I/2$ has the eigenvalue $\tfrac12$ twice, in *every* basis, so $\log_2\sigma = \log_2(\tfrac12)\,I = -I$, and the cross term is $-\mathrm{tr}\,\rho\,(-I) = \mathrm{tr}\,\rho = 1$.
- Together, $S(|{+}\rangle\langle{+}|\,\|\,I/2) = 0 + 1 = 1$ bit.

The same two lines work for any $\rho$ in dimension $d$: $\log_2(I/d) = -\log_2 d\;I$, so $S(\rho\,\|\,I/d) = \log_2 d - S(\rho)$, the lift of `02/02`'s $D(p\|u) = \log_2 n - H(p)$. The von Neumann entropy of `01/13` is itself a relative entropy, measured from the maximally mixed state, up to the constant $\log_2 d$.

**The classical side.** Measure both states in the $Z$ basis: the Born probabilities are the diagonals, $(\tfrac12, \tfrac12)$ for both, so the classical relative entropy is $0$, the seed of `01/13`. Measure in the $X$ basis (Hadamard, then the diagonal): $|{+}\rangle$ gives $(1, 0)$, $I/2$ gives $(\tfrac12, \tfrac12)$, and $D = 1$ bit. The classical number depends on the measurement; the Umegaki number needs no choice. The cell checks each number, and the commuting case on the pair of `02/02`.

In [ ]:
# A function of a Hermitian matrix acts on its eigenvalues: log2 rho = V diag(log2 w) V^dagger.
rho_06 = density_from_bloch([0.6, 0.0, 0.0])
w06, V06 = np.linalg.eigh(rho_06)
print(f"Bloch (0.6, 0, 0): eigenvalues {np.round(w06, 4)};  log2 rho from the eigenbasis =\n{np.round((V06 @ np.diag(np.log2(w06)) @ V06.conj().T).real, 4)}")
print(f"np.log2 entry by entry, a different matrix =\n{np.round(np.log2(rho_06.real), 4)}")
print()

plus = density_matrix(KET_PLUS)          # rho   = |+><+|
mixed = maximally_mixed(2)               # sigma = I/2

# By hand: eigenvalues of each state, then the two terms of S = tr rho log rho - tr rho log sigma.
lam_rho, vec_rho = np.linalg.eigh(plus)
lam_sig = np.linalg.eigvalsh(mixed)
print(f"eigenvalues of |+><+| : {np.round(lam_rho, 12)}   eigenvector of the eigenvalue 1: {np.round(vec_rho[:, 1], 4)}  (= |+>, up to sign)")
print(f"eigenvalues of I/2    : {lam_sig}")
term_entropy = sum(l * np.log2(l) for l in lam_rho if l > 1e-12)        # 0 log 0 = 0
log2_sigma = np.log2(0.5) * np.eye(2)                                    # log2(I/2) = -I in every basis
term_cross = -np.real(np.trace(plus @ log2_sigma))
S_hand = term_entropy + term_cross
print(f"tr rho log2 rho = {term_entropy:.4f}   cross term -tr rho log2 sigma = {term_cross:.4f}   ->  S by hand = {S_hand:.4f} bit")
print(f"library quantum_relative_entropy(|+><+|, I/2) = {quantum_relative_entropy(plus, mixed):.4f} bit")
print()

# The classical side: Born probabilities of a Z measurement (the diagonal) and of an X measurement.
p_Z, q_Z = np.real(np.diag(plus)), np.real(np.diag(mixed))
p_X = np.real(np.diag(HADAMARD @ plus @ HADAMARD.conj().T)) + 0.0   # + 0.0 turns -0.0 into 0.0
q_X = np.real(np.diag(HADAMARD @ mixed @ HADAMARD.conj().T))
print(f"Z measurement: p = {np.round(p_Z, 4)}, q = {np.round(q_Z, 4)}   kl_divergence = {kl_divergence(p_Z, q_Z):.4f} bit")
print(f"X measurement: p = {np.round(p_X, 4) + 0.0}, q = {np.round(q_X, 4)}   kl_divergence = {kl_divergence(p_X, q_X):.4f} bit")
print()

# The identity S(rho || I/d) = log2 d - S(rho), on |+> and on a mixed qubit state.
rho_mixed = density_from_bloch([0.3, -0.4, 0.5])
for name, r in [("|+><+|", plus), ("Bloch (0.3, -0.4, 0.5)", rho_mixed)]:
    print(f"{name:24s}: S(rho || I/2) = {quantum_relative_entropy(r, mixed):.4f}   1 - S(rho) = {1 - von_neumann_entropy(r):.4f}")

# The commuting case: two diagonal states are two probability vectors (the pair of 02/02).
p, q = np.array([0.7, 0.2, 0.1]), np.array([0.4, 0.4, 0.2])
print(f"diagonal states diag{p.tolist()} vs diag{q.tolist()}: Umegaki = {quantum_relative_entropy(np.diag(p), np.diag(q)):.4f}   classical D(p||q) = {kl_divergence(p, q):.4f}")

**Read the output.** Built from the eigenbasis, $\log_2\rho$ has diagonal $-1.3219$ and off-diagonal $1$, as by hand; `np.log2` entry by entry gives $-1$ and $-1.737$, another matrix. For the seed, the eigenvalues are $(0, 1)$ and $(\tfrac12, \tfrac12)$, the two terms are $0$ and $1$, and the library returns the same $1$ bit. With `kl_divergence`, the $Z$ outcomes give $0$ and the $X$ outcomes $1$ bit. The identity $S(\rho\|I/2) = 1 - S(\rho)$ holds on the pure state and on a mixed one ($0.3991$ both ways). And on two diagonal states the Umegaki relative entropy is the classical $0.2651$ bits of `02/02`, to four decimals: a lift adds nothing in the commuting case.

So the one bit is what an $X$ measurement reveals and a $Z$ measurement throws away. Could some measurement reveal *more*? On the left of the figure, measure along an axis of the $x$–$z$ plane at angle $\theta$ from $\hat z$ and compute the classical relative entropy of the outcomes. On the right, walk the line $\rho_t = (1-t)\,|{+}\rangle\langle{+}| + t\,I/2$ of `01/13`, where the coherence drains away.

In [ ]:
def born_along(rho, theta):
    # outcome probabilities of a measurement along n = (sin theta, 0, cos theta): projectors (I +- n.sigma)/2
    n_sigma = np.sin(theta) * PAULI_X + np.cos(theta) * PAULI_Z
    p_up = np.real(np.trace(rho @ (np.eye(2) + n_sigma) / 2))
    return np.array([p_up, 1.0 - p_up])

thetas = np.linspace(0.0, np.pi, 181)
kl_axis = np.array([kl_divergence(born_along(plus, th), born_along(mixed, th)) for th in thetas])
S_seed = quantum_relative_entropy(plus, mixed)
print(f"largest classical relative entropy over the measurement axes: {kl_axis.max():.4f} bit at theta = {np.degrees(thetas[kl_axis.argmax()]):.0f} degrees (the X axis);  Umegaki: {S_seed:.4f} bit")
print(f"at theta = 0 (the Z axis): {kl_axis[0]:.4f} bit;  at theta = 45 degrees: {kl_axis[45]:.4f} bit")

ts = np.linspace(0.0, 1.0, 101)
S_line = np.array([quantum_relative_entropy((1 - t) * plus + t * mixed, mixed) for t in ts])
S_ident = np.array([1 - von_neumann_entropy((1 - t) * plus + t * mixed) for t in ts])
kl_Z_line = np.array([kl_divergence(np.real(np.diag((1 - t) * plus + t * mixed)), q_Z) for t in ts])
print(f"along the line: S(rho_t || I/2) at t = 0.5 is {S_line[50]:.4f} bit;  max |S - (1 - S(rho_t))| = {np.max(np.abs(S_line - S_ident)):.1e};  Z-diagonal KL stays at {kl_Z_line.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(np.degrees(thetas), kl_axis, color=viz.SOURCE_COLOR, lw=2.4, label="classical KL of the two outcome distributions")
ax.axhline(S_seed, color=COLORS["highlight"], lw=2.0, ls="--", label="Umegaki S(|+> || I/2) = 1 bit")
ax.set_xticks([0, 45, 90, 135, 180])
ax.set_xlabel("measurement axis: angle theta from the z axis (degrees)")
ax.set_ylabel("relative entropy (bits)")
ax.set_ylim(-0.05, 1.25)
ax.set_title("|+> vs I/2, measured along one axis", pad=10)
ax.legend(loc="upper left")
ax = axes[1]
ax.plot(ts, S_line, color=COLORS["quantum"], lw=2.4, label="Umegaki S(rho_t || I/2)")
ax.plot(ts[::10], S_ident[::10], "o", color=COLORS["text"], ms=5, label="1 - S(rho_t), from the identity")
ax.plot(ts, kl_Z_line, color=viz.TARGET_COLOR, lw=2.6, ls=":", label="classical KL of the Z diagonals")
ax.set_xlabel("mixing weight t  (t = 0: |+><+|,  t = 1: I/2)")
ax.set_ylabel("relative entropy (bits)")
ax.set_ylim(-0.05, 1.25)
ax.set_title("Along the line from |+><+| to I/2", pad=10)
ax.legend(loc="upper right")
plt.show()

**Read the figure.** On the left, the classical relative entropy of the measured outcomes is $0$ on the $Z$ axis, rises to $0.3991$ bits at $45°$ (the value of the mixed state above, not by chance: both come down to the probabilities $\tfrac12(1 \pm 1/\sqrt2)$), and reaches the full $1$ bit only on the $X$ axis, the one axis along which $|{+}\rangle$ points. It never rises above the dashed Umegaki line, and section 3 derives from Lindblad's theorem that no measurement in a basis can do better. On the right, the Umegaki relative entropy falls from $1$ bit to $0$ ($0.1887$ bits at $t = \tfrac12$), with the dots of $1 - S(\rho_t)$ on the curve, as the identity says. The classical line from the $Z$ diagonals is $0$ the whole way: a diagonal-only view never sees the coherence drain out. Section 3 meets $0.1887$ again.

## 2. Never negative, sometimes infinite

**Klein's inequality.** For any two density matrices, $S(\rho\|\sigma) \ge 0$, with equality if and only if $\rho = \sigma$. This is a theorem. Its root is a trace inequality from O. Klein (1931); the general form for convex $f$ and Hermitian $A, B$, $\mathrm{tr}\,[f(A) - f(B) - (A - B)f'(B)] \ge 0$, now carries his name. With $f(t) = t\ln t$, $A = \rho$, $B = \sigma$, the trace terms cancel and what remains is the Umegaki relative entropy in nats; today "Klein's inequality" usually names this consequence (Nielsen and Chuang, §11.3.1).

This is the densest step of the notebook, and the one that turns a classical proof you own into a quantum one: Nielsen and Chuang's argument needs only `02/02`'s Gibbs inequality and one concavity step. Diagonalise $\rho = \sum_i p_i|i\rangle\langle i|$ and $\sigma = \sum_j q_j|j\rangle\langle j|$, and collect the overlaps $P_{ij} = |\langle i|j\rangle|^2$: non-negative, with every row and column summing to $1$. In the eigenbasis of $\rho$, $\mathrm{tr}\,\rho\log_2\sigma = \sum_i p_i\,\langle i|\log_2\sigma|i\rangle$, and $\langle i|\log_2\sigma|i\rangle = \sum_j \log_2 q_j\,|\langle i|j\rangle|^2 = \sum_j P_{ij}\log_2 q_j$, so

$$
S(\rho\|\sigma) \;=\; \sum_i p_i\log_2 p_i \;-\; \sum_i p_i \sum_j P_{ij}\log_2 q_j .
$$

Each inner sum is a weighted average of $\log_2 q_j$, and the logarithm is concave, so by Jensen (`02/01`) it is at most $\log_2 r_i$ with $r_i = \sum_j P_{ij} q_j$. Because the columns of $P$ sum to $1$, $r$ is a probability vector, and

$$
S(\rho\|\sigma) \;\ge\; \sum_i p_i\log_2\frac{p_i}{r_i} \;=\; D(p\,\|\,r) \;\ge\; 0
$$

by Gibbs' inequality: the quantum relative entropy is at least the classical one between the eigenvalues of $\rho$ and the probabilities $\sigma$ gives to the eigenvectors of $\rho$. The equality cases of the two steps give $\rho = \sigma$ (Nielsen and Chuang write them out). If $\sigma$ is singular, either $\rho$ has weight on its kernel and $S = +\infty$ (below), or $P_{ij} = 0$ whenever $p_i > 0$ and $q_j = 0$, and the argument runs over the remaining terms.

**A worked example.** Take $\rho$ with Bloch vector $(0.6, 0, 0)$ and $\sigma$ with $(0, 0, 0.6)$. Both have eigenvalues $\tfrac{1 \pm 0.6}{2} = (0.8, 0.2)$, with eigenvectors $|{\pm}\rangle$ for $\rho$ and $|0\rangle, |1\rangle$ for $\sigma$. Every overlap is $\tfrac12$, so $r = (\tfrac12, \tfrac12)$. The entropy term is $-h(0.8)$ ($h$ the binary entropy of `02/01`) and each inner average is $\tfrac12(\log_2 0.8 + \log_2 0.2)$. The cell computes each number, then tries two thousand random pairs.

In [ ]:
rho_ex = density_from_bloch([0.6, 0.0, 0.0])
sig_ex = density_from_bloch([0.0, 0.0, 0.6])
p_ex, U_rho = np.linalg.eigh(rho_ex)
q_ex, U_sig = np.linalg.eigh(sig_ex)
P = np.abs(U_rho.conj().T @ U_sig) ** 2                                  # P_ij = |<i|j>|^2
r_ex = P @ q_ex
print(f"eigenvalues of rho: {np.round(p_ex, 4)}   of sigma: {np.round(q_ex, 4)}")
print(f"overlap matrix P =\n{np.round(P, 4)}\nrow sums {np.round(P.sum(1), 4)}, column sums {np.round(P.sum(0), 4)};  r = P q = {np.round(r_ex, 4)}")
entropy_term = np.sum(p_ex * np.log2(p_ex))
inner = P @ np.log2(q_ex)                                                # sum_j P_ij log2 q_j, for each i
print(f"entropy term sum p log2 p = -h(0.8) = {entropy_term:.4f};  inner averages sum_j P_ij log2 q_j = {np.round(inner, 4)}  <=  log2 r_i = {np.round(np.log2(r_ex), 4)}")
S_ex_hand = entropy_term - np.sum(p_ex * inner)
print(f"S by hand = {entropy_term:.4f} + {-np.sum(p_ex * inner):.4f} = {S_ex_hand:.4f} bit;  library: {quantum_relative_entropy(rho_ex, sig_ex):.4f};  lower bound D(p || r) = {kl_divergence(p_ex, r_ex):.4f}")
print(f"arguments swapped: S(sigma || rho) = {quantum_relative_entropy(sig_ex, rho_ex):.4f}")
print()

def random_state(rmax=0.95):
    # a qubit state with a random Bloch vector of length below rmax (full rank, so every S is finite)
    v = rng.normal(size=3)
    return density_from_bloch(v / np.linalg.norm(v) * rmax * rng.uniform() ** (1 / 3))

n_pairs = 2000
S_rand, gap_rand = [], []
for _ in range(n_pairs):
    a, b = random_state(), random_state()
    pa, Ua = np.linalg.eigh(a); qb, Ub = np.linalg.eigh(b)
    S_ab = quantum_relative_entropy(a, b)
    S_rand.append(S_ab)
    gap_rand.append(S_ab - kl_divergence(pa, (np.abs(Ua.conj().T @ Ub) ** 2) @ qb))
S_rand = np.array(S_rand)
print(f"{n_pairs} random pairs: smallest S(rho||sigma) = {S_rand.min():.2e};  smallest S - D(p||r) = {min(gap_rand):.2e}  (never negative)")
print(f"S(rho||rho) on the worked example: {quantum_relative_entropy(rho_ex, rho_ex):.1e}")

**Read the output.** The eigenvalues are $(0.2, 0.8)$ for both, every overlap is $0.5$, and $r = (0.5, 0.5)$. The entropy term is $-0.7219$; each inner average is $-1.3219$, below $\log_2 r_i = -1$: the Jensen step at work. The total, $-0.7219 + 1.3219 = 0.6000$ bits, is the library's value, above the lower bound $D(p\|r) = 0.2781$, itself above $0$. Over two thousand random pairs the value is never negative nor below its Gibbs bound, and a state against itself gives $0$. These numbers illustrate the theorem; the proof is the argument above, and the theorem is Klein's. Swap the arguments and you get $0.6000$ again. That is a feature of this pair, not of the measure: the Hadamard turns $\rho$ into $\sigma$ and $\sigma$ into $\rho$, and section 3 shows that rotating both states together changes nothing. In general the two orders differ. The pair $I/2$ and $|{+}\rangle\langle{+}|$ gives $1$ bit one way and $+\infty$ the other.

**When is it infinite?** In `02/02`, $D = +\infty$ as soon as $p$ put mass where $q$ put none. The quantum condition is the same idea with subspaces. The **support** of a state is the span of its eigenvectors with non-zero eigenvalue; its **kernel** is the span of the others. Then

$$
S(\rho\|\sigma) = +\infty \quad\text{if and only if}\quad \mathrm{supp}\,\rho \not\subseteq \mathrm{supp}\,\sigma .
$$

The reason is the cross term: on the kernel of $\sigma$, $\log_2\sigma$ has eigenvalue $-\infty$, so if the "leak" $\mathrm{tr}\,(\Pi_{\ker\sigma}\,\rho)$ is positive ($\Pi_{\ker\sigma}$ the projector onto the kernel), $-\mathrm{tr}\,\rho\log_2\sigma$ diverges. If the leak is zero, every logarithm the formula meets is finite. This is the test the library runs.

Disjoint supports, like those of $|0\rangle\langle0|$ and $|1\rangle\langle1|$, are a special case. Take $\rho = I/2$ and $\sigma = |{+}\rangle\langle{+}|$: the support of $\sigma$, the line through $|{+}\rangle$, lies *inside* that of $\rho$, the whole plane. The supports share $|{+}\rangle$, yet $\rho$ puts weight $\tfrac12$ on $|{-}\rangle$, the kernel of $\sigma$, so $S(I/2\,\|\,|{+}\rangle\langle{+}|) = +\infty$, while the reverse is the finite $1$ bit of section 1. Conversely, a singular $\sigma$ alone does not make the value infinite: on a qutrit, $|{+}\rangle\langle{+}|$ (in the span of $|0\rangle, |1\rangle$) against $\mathrm{diag}(\tfrac12, \tfrac12, 0)$ is finite. The cell computes each leak and value, the classical overlapping case with `kl_divergence`, and the infinite value as a limit, moving $\sigma_\varepsilon = (1-\varepsilon)\,|{+}\rangle\langle{+}| + \varepsilon\,I/2$ toward the pure state.

In [ ]:
def leak(rho, sigma, atol=1e-12):
    # weight that rho puts on the kernel of sigma: tr(Pi_ker(sigma) rho)
    w, V = np.linalg.eigh(sigma)
    K = V[:, w <= atol]
    return float(np.real(np.trace(K.conj().T @ rho @ K))) if K.shape[1] else 0.0

ket0_3, ket1_3 = np.eye(3)[0], np.eye(3)[1]
plus_3 = density_matrix((ket0_3 + ket1_3) / np.sqrt(2))                 # |+> inside a qutrit
cases = [
    ("|0><0| vs |1><1|  (supports do not intersect)", density_matrix(KET_0), density_matrix(KET_1)),
    ("I/2 vs |+><+|     (supports overlap, not contained)", mixed, plus),
    ("|+><+| vs I/2     (contained)", plus, mixed),
    ("qutrit |+><+| vs diag(1/2,1/2,0)  (contained, sigma singular)", plus_3, np.diag([0.5, 0.5, 0.0]).astype(complex)),
]
for name, r, s in cases:
    print(f"{name:62s} leak = {leak(r, s):.2f}   S = {quantum_relative_entropy(r, s)}")
p3, q3 = np.array([0.5, 0.5, 0.0]), np.array([0.0, 0.5, 0.5])
print(f"classical, overlapping supports: kl_divergence({p3}, {q3}) = {kl_divergence(p3, q3)}")

eps = np.logspace(-8, 0, 81)
S_toward = np.array([quantum_relative_entropy(mixed, (1 - e) * plus + e * mixed) for e in eps])
S_away = np.array([quantum_relative_entropy((1 - e) * plus + e * mixed, mixed) for e in eps])
closed = -1 - 0.5 * np.log2(1 - eps / 2) - 0.5 * np.log2(eps / 2)    # by hand: eigenvalues of sigma_eps are 1 - eps/2 and eps/2
print(f"S(I/2 || sigma_eps) at eps = 1e-2, 1e-4, 1e-8: {np.round(S_toward[[60, 40, 0]], 3)};  closed form {np.round(closed[[60, 40, 0]], 3)}")
print(f"S(sigma_eps || I/2) at the same eps: {np.round(S_away[[60, 40, 0]], 4)}  (bounded by 1 bit)")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.hist(S_rand, bins=40, color=COLORS["quantum"], alpha=0.85)
ax.axvline(0.0, color=COLORS["text"], lw=1.2)
ax.axvline(S_ex_hand, color=COLORS["highlight"], lw=2.0, ls="--", label=f"worked example: {S_ex_hand:.3f} bit")
ax.set_xlabel("S(rho || sigma)  (bits)")
ax.set_ylabel("number of random pairs")
ax.set_title(f"Klein: {n_pairs} random qubit pairs, none below 0", pad=10)
ax.legend(loc="upper right")
ax = axes[1]
ax.plot(eps, S_toward, color=viz.SOURCE_COLOR, lw=2.4, label="S(I/2 || sigma_eps): rho leaks onto |->")
ax.plot(eps, S_away, color=viz.TARGET_COLOR, lw=2.4, label="S(sigma_eps || I/2): contained")
ax.set_xscale("log")
ax.set_xlabel("eps  (sigma_eps -> |+><+| as eps -> 0)")
ax.set_ylabel("relative entropy (bits)")
ax.set_title("The infinite value, approached as a limit", pad=10)
ax.legend(loc="upper right")
plt.show()

**Read the output and the figure.** $|0\rangle$ against $|1\rangle$ leaks its whole weight: infinite. The overlapping pair $I/2$ against $|{+}\rangle\langle{+}|$ leaks $0.50$ and is infinite too; reversed, it leaks nothing and gives $1$ bit. The qutrit example has a singular $\sigma$, no leak, and $1$ bit. The classical overlapping case, $(\tfrac12, \tfrac12, 0)$ against $(0, \tfrac12, \tfrac12)$, is infinite by `kl_divergence`: in both worlds the condition is containment of supports, which are subspaces in the quantum one.

On the left, the two thousand Klein values crowd between $0$ and $2$ bits with a long tail; none falls below $0$, and the worked example sits at $0.600$. On the right, as $\sigma_\varepsilon$ approaches $|{+}\rangle\langle{+}|$, the relative entropy from $I/2$ grows, for small $\varepsilon$, like $\tfrac12\log_2(1/\varepsilon)$: $2.826$ bits at $\varepsilon = 10^{-2}$, $6.144$ at $10^{-4}$, $12.788$ at $10^{-8}$, matching the closed form from the eigenvalues $1 - \varepsilon/2$ and $\varepsilon/2$. The other direction stays below $1$ bit. The infinite value is the end of this climb, the sharpest form of the asymmetry of `02/02`.

## 3. No channel can increase it: monotonicity

In `02/02b` (§3) you proved the data-processing inequality, $D(Tp\|Tq) \le D(p\|q)$ for any stochastic map $T$. A quantum channel (`01/19`) is the quantum stochastic map, so a quantum relative entropy worth the name must satisfy

$$
S\big(\Phi(\rho)\,\big\|\,\Phi(\sigma)\big) \;\le\; S(\rho\,\|\,\sigma) \qquad\text{for every channel } \Phi .
$$

**This is a theorem, and a deep one.** Lindblad proved it for the Umegaki relative entropy and completely positive trace-preserving maps in 1975; Uhlmann extended it to a far more general setting in 1977. The classical proof of `02/02b` worked outcome by outcome, which matrices that do not commute do not allow; the quantum proof needs operator concavity theorems this course does not develop. We quote the theorem, check it on the channels of `01/19`, and derive three consequences.

**A check by hand.** Send the seed pair through the depolarizing channel $\Phi_p(\rho) = (1-p)\,\rho + p\,I/2$ with $p = \tfrac12$. $I/2$ is a fixed point; the Bloch vector of $|{+}\rangle\langle{+}|$ shrinks to length $\tfrac12$, so its eigenvalues become $(0.75, 0.25)$, and by section 1, $S = 1 - h(0.75)$, below the $1$ bit you started from. The second channel is dephasing, with the Kraus operators $K_0 = \mathrm{diag}(1, \sqrt{1-\lambda})$, $K_1 = \mathrm{diag}(0, \sqrt\lambda)$ of `01/19`: it keeps the diagonal and multiplies the coherence by $\sqrt{1-\lambda}$. The library has no dephasing function, so the cell builds it and checks it with `is_cptp`.

**Three consequences.**
- *Unitaries change nothing.* A unitary and its inverse are both channels, so the theorem applied both ways gives $S(U\rho U^\dagger\,\|\,U\sigma U^\dagger) = S(\rho\|\sigma)$: the value does not depend on the basis, provided both states are rotated together.
- *No measurement reveals more.* Measuring in a basis $\{|k\rangle\}$ and keeping the outcome is the channel with Kraus operators $|k\rangle\langle k|$, which leaves only the Born probabilities on the diagonal. So the classical relative entropy of any such measurement is at most $S(\rho\|\sigma)$: the ceiling of the first figure. Complete dephasing in $Z$ is the $Z$ measurement.
- *Noise is a one-way street.* Two depolarizing channels in a row are one, with the Bloch vector shrunk by $(1-p_1)(1-p_2)$, so stronger noise is weaker noise followed by more, and $S(\Phi_p(\rho)\|\Phi_p(\sigma))$ can never rise as $p$ grows. The same holds for dephasing.

In [ ]:
def dephasing_channel(lam):
    # Kraus operators of 01/19: the diagonal is kept, the coherence is multiplied by sqrt(1 - lam)
    return [np.diag([1.0, np.sqrt(1.0 - lam)]).astype(complex), np.diag([0.0, np.sqrt(lam)]).astype(complex)]

print(f"is_cptp: depolarizing(0.5) {is_cptp(depolarizing_channel(0.5))},  dephasing(0.3) {is_cptp(dephasing_channel(0.3))}")
dep = depolarizing_channel(0.5)
out_plus, out_mixed = apply_channel(plus, dep), apply_channel(mixed, dep)
print(f"depolarizing p = 0.5: eigenvalues of the output of |+><+| = {np.round(np.linalg.eigvalsh(out_plus), 4)};  output of I/2 = I/2: {np.allclose(out_mixed, mixed)}")
print(f"   S before = {quantum_relative_entropy(plus, mixed):.4f}   S after, by hand 1 - h(0.75) = {1 - shannon_entropy([0.75, 0.25]):.4f}   library = {quantum_relative_entropy(out_plus, out_mixed):.4f}   (h(0.75) = {shannon_entropy([0.75, 0.25]):.4f})")
print(f"   on the pair of section 2: S before = {quantum_relative_entropy(rho_ex, sig_ex):.4f}   after = {quantum_relative_entropy(apply_channel(rho_ex, dep), apply_channel(sig_ex, dep)):.4f}")
deph_full = dephasing_channel(1.0)
r_d, s_d = apply_channel(rho_ex, deph_full), apply_channel(sig_ex, deph_full)
print(f"complete dephasing (lambda = 1) on the pair of section 2: S after = {quantum_relative_entropy(r_d, s_d):.4f};  classical KL of the Z diagonals {np.round(np.real(np.diag(rho_ex)), 2)} vs {np.round(np.real(np.diag(sig_ex)), 2)} = {kl_divergence(np.real(np.diag(rho_ex)), np.real(np.diag(sig_ex))):.4f}")
print(f"complete dephasing on the seed pair: S after = {quantum_relative_entropy(apply_channel(plus, deph_full), apply_channel(mixed, deph_full)):.4f}  (the Z-measurement view of section 1)")
print()

# Unitary invariance: rotate both states together, then only one.
U = rz(0.7) @ ry(1.9) @ rz(-0.4)
rot = lambda A: U @ A @ U.conj().T
print(f"rotate both:  S = {quantum_relative_entropy(rot(rho_ex), rot(sig_ex)):.4f}  (before {quantum_relative_entropy(rho_ex, sig_ex):.4f});  rotate rho only: S = {quantum_relative_entropy(rot(rho_ex), sig_ex):.4f}")

# Random check: random full-rank pairs, a random strength of each channel, a random rotation.
worst = {"depolarizing": -np.inf, "dephasing": -np.inf, "rotation (|change|)": 0.0}
for _ in range(n_pairs):
    a, b = random_state(), random_state()
    S0 = quantum_relative_entropy(a, b)
    Kd, Kf = depolarizing_channel(rng.uniform()), dephasing_channel(rng.uniform())
    worst["depolarizing"] = max(worst["depolarizing"], quantum_relative_entropy(apply_channel(a, Kd), apply_channel(b, Kd)) - S0)
    worst["dephasing"] = max(worst["dephasing"], quantum_relative_entropy(apply_channel(a, Kf), apply_channel(b, Kf)) - S0)
    V = rz(rng.uniform(0, 2 * np.pi)) @ ry(rng.uniform(0, np.pi)) @ rz(rng.uniform(0, 2 * np.pi))
    worst["rotation (|change|)"] = max(worst["rotation (|change|)"], abs(quantum_relative_entropy(V @ a @ V.conj().T, V @ b @ V.conj().T) - S0))
for k, v in worst.items():
    print(f"{n_pairs} random pairs, {k:20s}: largest S(after) - S(before) = {v:.1e}")

**Read the output.** Both channels pass `is_cptp`. Depolarizing at $p = \tfrac12$ gives eigenvalues $(0.25, 0.75)$, leaves $I/2$ in place, and the relative entropy falls from $1$ bit to $1 - 0.8113 = 0.1887$, by hand and by the library. This is the $0.1887$ of the first figure: the output is $\rho_t$ at $t = \tfrac12$. On the pair of section 2 the same channel takes $0.6000$ bits to $0.1340$. Complete dephasing takes that pair to $0.3219$, the classical relative entropy of its $Z$ diagonals $(0.5, 0.5)$ and $(0.8, 0.2)$, and the seed pair to $0$: dephasing and measuring are the same channel. A common rotation leaves $0.6000$ unchanged; rotating only one state gives $1.1230$, a different pair. Over two thousand random pairs neither channel ever increased the value, and a common rotation moved it by at most $10^{-14}$, rounding. The numbers check Lindblad's theorem on one family of examples; the theorem covers every channel in every dimension.

The figure follows the value as the noise grows.

In [ ]:
ps = np.linspace(0.0, 1.0, 101)
pairs = [("|+> vs I/2", plus, mixed, COLORS["highlight"]), ("Bloch (0.6,0,0) vs (0,0,0.6)", rho_ex, sig_ex, COLORS["quantum"])]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
deph_curves = []
for name, r, s, col in pairs:
    S_dep = [quantum_relative_entropy(apply_channel(r, depolarizing_channel(x)), apply_channel(s, depolarizing_channel(x))) for x in ps]
    S_deph = [quantum_relative_entropy(apply_channel(r, dephasing_channel(x)), apply_channel(s, dephasing_channel(x))) for x in ps]
    print(f"{name:30s}: largest step up along p: {np.max(np.diff(S_dep)):.1e};  along lambda: {np.max(np.diff(S_deph)):.1e};  value at lambda = 1: {S_deph[-1]:.4f}")
    axes[0].plot(ps, S_dep, color=col, lw=2.4, label=name)
    axes[1].plot(ps, S_deph, color=col, lw=2.4, label=name)
    deph_curves.append(np.array(S_deph))
    axes[1].plot([1.0], [kl_divergence(np.real(np.diag(r)), np.real(np.diag(s)))], "o", color=COLORS["text"], ms=6)
print(f"dephasing: the second pair's curve passes above the seed pair's from lambda = {ps[np.argmax(deph_curves[0] < deph_curves[1])]:.2f}")
axes[0].set_xlabel("depolarizing strength p"); axes[0].set_ylabel("S(Phi(rho) || Phi(sigma))  (bits)")
axes[0].set_title("Depolarizing: (1 - p) rho + p I/2", pad=10); axes[0].legend(loc="upper right")
axes[1].set_xlabel("dephasing strength lambda"); axes[1].set_ylabel("S(Phi(rho) || Phi(sigma))  (bits)")
axes[1].set_title("Dephasing (dots: KL of the Z diagonals)", pad=10); axes[1].legend(loc="upper right")
for ax in axes:
    ax.set_ylim(-0.05, 1.1)
plt.show()

**Read the figure.** Every curve starts at the untouched value ($1$ bit and $0.6$ bit) and only steps down. Depolarizing drives both pairs to $0$ at $p = 1$, where every state is $I/2$. Dephasing stops the curves at the black dots, the classical relative entropy of the $Z$ diagonals: $0$ for the seed pair, whose whole difference was coherence, and $0.3219$ bits for the second pair, which keeps its population difference. The drop from start to dot is the distinguishability carried by the off-diagonal entries: a channel can spend it, never create it. Under dephasing the two curves cross near $\lambda = 0.42$. Monotonicity is a statement about each pair through the same channel; it says nothing about the order between two pairs.

## 4. Where the quantum relative entropy goes

- **Quantum mutual information** (`02/10`): $I(A{:}B) = S(\rho_{AB}\,\|\,\rho_A\otimes\rho_B)$, as `02/03` built $I(X;Y) = D(p_{XY}\|p_X\otimes p_Y)$. Klein makes it non-negative; monotonicity under one-sided channels says local processing cannot create correlation.
- **The Bures distance** (`02/12`). Relative entropy is not a distance; `02/12` builds one from the fidelity of `01/14`, which also sees the coherence separating $|{+}\rangle$ from $I/2$.
- **Entropic quantum optimal transport** (`04/06`–`04/08`). The formula $\mathrm{tr}\,\rho(\log\rho - \log K)$ still makes sense when $K = \exp(-C/\varepsilon)$ is a positive operator that is not a state. With it, the entropic transport objective $\mathrm{tr}(C\rho_{AB}) - \varepsilon S(\rho_{AB})$ equals $\varepsilon$ times that quantity, in nats, for every coupling $\rho_{AB}$. Because $K$ is not a state, Klein's inequality does not apply to it: the value can be negative, and normalising $K$ shifts it by the constant $\varepsilon\ln\mathrm{tr}\,K$, which does not change which coupling is smallest. Module 04 builds on that identity.
- **Units.** Pass `base=np.e` for nats, the unit of module 04; a value in nats is the value in bits times $\ln 2$.

## Your turn

1. **The other direction, by hand.** Compute $S(\sigma_\varepsilon\,\|\,I/2)$ from the eigenvalues of $\sigma_\varepsilon$ and the identity of section 1, and check it against the curve labelled "S(sigma_eps || I/2): contained" in the second figure. What is its limit as $\varepsilon \to 0$, and why is it finite?
2. **Commuting, but not diagonal.** Draw $p, q$ with `rng.dirichlet(np.ones(4))` and a random unitary with `Q, _ = np.linalg.qr(rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4)))`, and check `np.allclose(Q.conj().T @ Q, np.eye(4))`. (a) $Q\,\mathrm{diag}(p)\,Q^\dagger$ and $Q\,\mathrm{diag}(q)\,Q^\dagger$ commute but are not diagonal: check that their Umegaki value is still $D(p\|q)$, and say which consequence of section 3 explains it. (b) Rotate only the first one: compute the overlap matrix $P$ and the lower bound $D(p\|r)$ of section 2, and check $S \ge D(p\|r)$.
3. **Neither support inside the other.** On a qutrit, take $\rho = \mathrm{diag}(\tfrac12, \tfrac12, 0)$ and $\sigma = \tfrac12|0\rangle\langle 0| + \tfrac12|v\rangle\langle v|$, $|v\rangle = (|1\rangle + |2\rangle)/\sqrt2$. Find each support and kernel by hand, predict whether $S(\rho\|\sigma)$ and $S(\sigma\|\rho)$ are finite, and check with the library and `leak`.
4. **Harder: the measurement ceiling.** For qubits, the classical relative entropy along an axis $\hat n$ is $D(\hat n) = D\big(P_\rho(\hat n)\,\|\,P_\sigma(\hat n)\big)$ with $P_\rho(\pm\hat n) = \tfrac12(1 \pm \vec r_\rho\cdot\hat n)$. (a) For the pair of section 2, maximise $D(\hat n)$ over a grid covering the sphere and compare with $S = 0.6$. (b) For two Bloch vectors along the same axis, show the maximum *equals* $S$; where is it reached? (c) Explain (a) with Lindblad's second consequence and (b) with section 1. No measurement along an axis collects the full $0.6$.

## What you built

- You can compute the Umegaki relative entropy by hand from eigenvalues: $1$ bit for $|{+}\rangle$ against $I/2$, where the classical $Z$ statistics give $0$.
- You can recognise $S(\rho\|I/d) = \log_2 d - S(\rho)$ and the classical $D(p\|q)$ hidden in any commuting pair.
- You can prove Klein's inequality from Gibbs' and check its bound on an example ($0.600 \ge 0.278 \ge 0$).
- You can state when the value is infinite, test it with the leak onto the kernel of $\sigma$, and give an infinite example with overlapping supports.
- You can state Lindblad's theorem, check it on two channels of `01/19`, and derive unitary invariance and the measurement ceiling.

The seed you planted in `01/13` now has its relative entropy, with the three properties that make it trustworthy.

## What's next

In `02/10_quantum_mutual_information` you point today's quantity at a two-qubit state and the product of its marginals, and find that a Bell pair shares $2$ bits, twice what two classical bits can share.

## References

- H. Umegaki, *Conditional expectation in an operator algebra, IV (entropy and information)*, Kodai Math. Sem. Rep. **14**, 59–85 (1962) — the definition of the quantum relative entropy. doi:10.2996/kmj/1138844604
- O. Klein, *Zur quantenmechanischen Begründung des zweiten Hauptsatzes der Wärmelehre*, Z. Phys. **72**, 767–775 (1931) — the trace inequality for convex functions behind $S(\rho\|\sigma) \ge 0$. doi:10.1007/BF01341997
- G. Lindblad, *Completely positive maps and entropy inequalities*, Commun. Math. Phys. **40**, 147–151 (1975) — monotonicity of the Umegaki relative entropy under completely positive trace-preserving maps. doi:10.1007/BF01609396
- A. Uhlmann, *Relative entropy and the Wigner–Yanase–Dyson–Lieb concavity in an interpolation theory*, Commun. Math. Phys. **54**, 21–32 (1977) — monotonicity in full generality. doi:10.1007/BF01609834
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 11, §11.3 (the quantum relative entropy and Klein's inequality, with the proof followed in section 2), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- V. Vedral, *The role of relative entropy in quantum information theory*, Rev. Mod. Phys. **74**, 197–234 (2002) — a review of the quantity, its properties and its uses. doi:10.1103/RevModPhys.74.197

**Previous:** `notebooks/02_InformationAndGeometry/08_two_geometries_one_simplex.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/10_quantum_mutual_information.ipynb`